[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_21_Agent_Frameworks_CrewAI_AutoGen_LangGraph.ipynb)

# Lesson 21 — Agent Frameworks: CrewAI vs. AutoGen vs. LangGraph
**Phase 3 · Choosing & comparing opinionated agent stacks**

You have now hand-built every layer of an agent system from scratch:
- L3–4: tool loops
- L5: memory
- L6: multi-agent orchestration
- L7/L20: RAG + vector infra
- L10: structured outputs
- L11: LangGraph stateful graphs
- L14: MCP
- L16–19: deployment, evals, security, streaming

That's a *lot* of plumbing. In the real world, most teams don't write a ReAct loop from scratch — they adopt a **framework** that bundles these patterns into a higher-level API. The big three in 2026:

| Framework | Mental model | Sweet spot |
|---|---|---|
| **LangGraph** | Explicit state machine / graph (low-level, you draw the edges) | Production agents that need control, observability, persistence |
| **CrewAI** | "Crew of roles" doing sequential / hierarchical tasks | Quickly prototyping role-play workflows (research team, content team) |
| **AutoGen** (v0.4+) | Conversational agents that **talk to each other** | Code-gen, problem-solving, agent-to-agent debate |

Today's mission is **not** "learn the API of one framework." It's:
1. Internalize the **three mental models** so you can read any framework's docs in 10 min.
2. Build the **same task** (researcher → critic → revised brief) in all three, side-by-side, so the trade-offs become obvious.
3. Walk away with a **decision framework**: when to adopt vs. build bespoke.

> 💡 Java analogy: this is the Spring vs. Micronaut vs. Quarkus debate, vector edition. They all solve the same problems, with different opinions on where to put the abstractions.

---

## Where this fits

| Lesson | What it gave you |
|---|---|
| L6 — Multi-agent | The patterns: sequential pipeline, orchestrator+subagents, critic loop, parallel fan-out |
| L11 — LangGraph | Stateful graphs from scratch (deep dive into one framework) |
| **L21 — TODAY** | **Comparison + decision framework across the major frameworks** |
| L22 — Cost Engineering | Routing, caching, batching across whichever stack you pick |
| L23 — Capstone | AutoResearcher v1.0 wired with the framework you chose |


## 0. Setup

Run this once. We install three frameworks in the same Colab. Total install ≈ 2-3 minutes (first time).

**Colab Secrets needed:**
- `ANTHROPIC_API_KEY` — all three frameworks call Claude

Optional:
- `OPENAI_API_KEY` — only if you swap models in the 💡 EXPERIMENT cells


In [ ]:
# Install all three frameworks. ~2-3 min on Colab first run.
!pip install -q \
    anthropic \
    crewai \
    autogen-agentchat \
    "autogen-ext[anthropic]" \
    langgraph \
    langchain-anthropic


In [ ]:
import os, asyncio, json, time
from typing import TypedDict, List, Dict, Any, Optional

# ---- API keys (Colab Secrets or os.environ) ----
def load_key(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        try: return userdata.get(name)
        except Exception: pass
    except ImportError:
        pass
    return os.environ.get(name)

ANTHROPIC_API_KEY = load_key("ANTHROPIC_API_KEY")
assert ANTHROPIC_API_KEY, "Set ANTHROPIC_API_KEY in Colab Secrets"
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# Frameworks pick up the key from env automatically.
MODEL = "claude-sonnet-4-5"   # adjust if you have access to a different family
print("Anthropic key:", "OK" if ANTHROPIC_API_KEY else "MISSING")
print("Model:        ", MODEL)


---
## 1. What an "agent framework" actually does

When you wrote your own ReAct loop in Lesson 4, you handled, by hand, all of this:

1. **System prompt construction** — role, goal, constraints
2. **Tool registration** — JSON schemas, dispatching results back
3. **The loop** — send → parse → execute → re-send, until "done"
4. **State** — what the agent has seen, what it has produced
5. **Multi-agent coordination** — when to hand off, who speaks next
6. **Termination** — when to stop (max steps? specific token? converged?)
7. **Observability** — what happened, in what order, how much it cost

An **agent framework** is a library that bundles some subset of those into an API. The frameworks differ in **which abstraction sits at the center**:

- **LangGraph** centers on **state + nodes + edges**. You explicitly draw "after researcher runs, go to critic; if critique says APPROVE, end; else loop." Lowest level, most control. Closest to what you wrote in L11.
- **CrewAI** centers on **roles and tasks**. You describe "a Researcher, a Critic, a Writer; here are their tasks in order." It plans the conversation under the hood.
- **AutoGen** centers on **agents that converse**. You drop agents into a group chat, set a termination condition, and let them talk. Most "emergent" behavior; least explicit control.

A useful way to plot them — control vs. ergonomics:

```
                  more control / more code
                         ↑
                    ┌── LangGraph ──┐
                    │               │
                    │               │
       hand-rolled ─┼─               ─┼─ CrewAI
                    │               │
                    │               │
                    └── AutoGen ───┘
                         ↓
                  more emergent / less code
```

There is **no winner**. They're tools for different ends. Today's notebook proves that with the same task in all three.


---
## 2. The shared task

To compare frameworks apples-to-apples we'll implement the **same** thing in each:

> **Researcher → Critic → Revised Brief**
>
> 1. **Researcher** writes a ~150-word brief on a topic.
> 2. **Critic** reviews. Either replies `APPROVE` or returns 2-3 specific edit requests.
> 3. If critic says APPROVE → stop. Else researcher revises (up to 3 iterations).
> 4. Final output: the approved brief.

This is the **critic-loop pattern** from Lesson 6. It's a great benchmark because it exercises all the moving parts: roles, multi-turn, state, termination, conditional flow.

We'll run all three on the same topic so you can compare outputs and code shape directly.


In [ ]:
TOPIC = "How tool use transforms a stateless LLM into a stateful agent — explain to a Java engineer new to AI"
MAX_ITERATIONS = 3


---
## 3. CrewAI — roles & tasks

**Mental model**: a **Crew** is a team of **Agents**, each with a **role / goal / backstory**. The crew works through **Tasks** in a `Process` (sequential or hierarchical). Each task has a `description`, an `agent`, and an `expected_output`. Tasks can `context` from prior tasks.

**Pros**
- Fastest to prototype "team of specialists" patterns. Reads like Jira tickets.
- Built-in tools, memory, planner, hierarchical manager-agent mode.
- Very approachable. Two screens of code → a working multi-agent flow.

**Cons**
- High-level → less control. Hard to enforce arbitrary state transitions.
- Iteration / looping is *not* its strong suit (the abstraction is "tasks in order", not "loop until").
- The "magic" can hide what's happening — you have to trust it more than LangGraph.

Let's build the researcher → critic loop in CrewAI. To get a real *loop*, we use CrewAI's `Process.hierarchical` mode with a manager agent — or, more transparently, we run the crew in a Python `for` loop and pass the prior critique back as context. The latter shows the seams clearly.


In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

# Shared LLM. CrewAI uses LiteLLM under the hood, so model strings are LiteLLM-format.
llm = LLM(model=f"anthropic/{MODEL}", temperature=0.3)

researcher = Agent(
    role="Senior AI Researcher",
    goal="Write crystal-clear, ~150-word technical briefs aimed at engineers new to AI.",
    backstory=(
        "You have 10 years explaining technical concepts to mid-career engineers. "
        "You favor concrete analogies (especially to Java/backend systems) over abstract jargon."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

critic = Agent(
    role="Technical Editor",
    goal="Ensure briefs are precise, well-structured, and useful to the target reader.",
    backstory=(
        "You are blunt and specific. You either APPROVE a brief that meets the bar, "
        "or you return exactly 2-3 concrete edit requests — never vague feedback."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

def run_crewai_loop(topic: str, max_iter: int = MAX_ITERATIONS) -> Dict[str, Any]:
    draft = ""
    critique = "(no prior critique — this is the first draft)"
    for i in range(1, max_iter + 1):
        # --- Researcher task ---
        research_task = Task(
            description=(
                f"Write a ~150-word brief on the following topic:\n\nTOPIC: {topic}\n\n"
                f"PRIOR CRITIQUE TO ADDRESS:\n{critique}\n\n"
                f"If this is iteration 1, ignore the critique. Otherwise, revise the prior draft to address it.\n"
                f"PRIOR DRAFT:\n{draft if draft else '(none yet)'}"
            ),
            agent=researcher,
            expected_output="A ~150-word brief in plain prose, no headings.",
        )
        crit_task = Task(
            description=(
                "Read the brief just written. If it is excellent for a Java engineer new to AI, "
                "respond with EXACTLY the single word: APPROVE. Otherwise, return 2-3 specific, "
                "concrete edit requests as a numbered list. No preamble."
            ),
            agent=critic,
            expected_output="Either 'APPROVE' or a numbered list of 2-3 edit requests.",
            context=[research_task],
        )
        crew = Crew(
            agents=[researcher, critic],
            tasks=[research_task, crit_task],
            process=Process.sequential,
            verbose=False,
        )
        result = crew.kickoff()
        # CrewAI stores per-task outputs on the result object.
        draft = str(result.tasks_output[0])
        critique = str(result.tasks_output[1]).strip()
        print(f"--- CrewAI iter {i} ---")
        print(f"[draft head] {draft[:120]}...")
        print(f"[critique]   {critique[:120]}...\n")
        if critique.strip().upper().startswith("APPROVE"):
            return {"final_draft": draft, "iterations": i, "status": "approved"}
    return {"final_draft": draft, "iterations": max_iter, "status": "max_iter_reached"}

crew_result = run_crewai_loop(TOPIC)
print("=== CREWAI RESULT ===")
print(f"status     : {crew_result['status']}")
print(f"iterations : {crew_result['iterations']}")
print(f"\nFINAL BRIEF:\n{crew_result['final_draft']}")

# 💡 EXPERIMENT: switch process to Process.hierarchical and define a manager_llm
#                to let CrewAI plan the loop itself. Note how much control you give up.
# 💡 EXPERIMENT: add allow_delegation=True on the researcher — it can now ask the critic
#                mid-draft. Watch the chatter grow.


**What you should observe**: CrewAI's task-graph is *sequential*. The loop only exists because we wrote a Python `for`. CrewAI itself doesn't have first-class "loop until condition" — its strength is "team does these N tasks in order".

This is the framework's opinion. It will feel *liberating* for "research team produces a report" workflows, and *restrictive* the moment you want self-correcting cycles.


---
## 4. AutoGen — agents that converse

**Mental model**: agents are **chat participants**. You drop them into a **GroupChat** (or `Team`, in v0.4+), set a **termination condition** (e.g., "stop when message contains APPROVE", "stop at 10 messages"), and call `team.run(task=...)`. The runtime decides who speaks next based on the team type (round-robin, selector, magentic).

**Pros**
- Beautiful for problem-solving / code-gen patterns where agents debate.
- Strong **code execution** support: `UserProxyAgent` can run code, install packages, fix its own errors. This is what Microsoft uses for autonomous coding demos.
- Async-native — fits FastAPI/streaming naturally.

**Cons**
- Async API can be a surprise if you're sync-minded.
- The "let agents figure it out" style can drift — you'll add termination guards.
- v0.4 was a major rewrite (`autogen_agentchat` package). Older tutorials use the v0.2 API and won't work. **Use v0.4+.**


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.anthropic import AnthropicChatCompletionClient

# v0.4+ uses pluggable model clients. Anthropic ships in autogen-ext.
model_client = AnthropicChatCompletionClient(model=MODEL)

ag_researcher = AssistantAgent(
    name="researcher",
    model_client=model_client,
    system_message=(
        "You write a ~150-word brief on the topic the user gives. "
        "If the critic returns edits, you revise the brief and post the new version. "
        "Do NOT include preamble like 'here is the brief'. Just the brief."
    ),
)

ag_critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    system_message=(
        "You critique briefs. After each new draft from the researcher, respond with EXACTLY "
        "the single word APPROVE if it's excellent for a Java engineer new to AI, otherwise "
        "give 2-3 specific edit requests as a numbered list. No other text."
    ),
)

# Stop when 'APPROVE' is said, OR after 8 messages — whichever first.
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

team = RoundRobinGroupChat([ag_researcher, ag_critic], termination_condition=termination)

async def run_autogen():
    result = await team.run(task=f"Write the brief on this topic: {TOPIC}")
    return result

ag_result = await run_autogen()

print("=== AUTOGEN MESSAGES ===")
for msg in ag_result.messages:
    src = getattr(msg, "source", "?")
    content = getattr(msg, "content", str(msg))
    if isinstance(content, list):  # multimodal payload
        content = " ".join(str(c) for c in content)
    print(f"\n[{src}] {str(content)[:300]}")

# Pull the last researcher message as the final brief.
final_brief = next(
    (str(m.content) for m in reversed(ag_result.messages) if getattr(m, "source", "") == "researcher"),
    "(no researcher message?)",
)
print("\n=== AUTOGEN FINAL BRIEF ===")
print(final_brief)

# 💡 EXPERIMENT: swap RoundRobinGroupChat for SelectorGroupChat — the selector LLM
#                picks who speaks next based on context. More emergent, less predictable.
# 💡 EXPERIMENT: add a third agent (e.g., 'fact_checker') and see how the dynamics change.


**What you should observe**: in AutoGen, *we never wrote the loop*. We just listed participants and a stop condition. The framework runs the conversation until the termination is hit. This is the "conversational" model — it's powerful for emergent collaboration, but you'll notice you have *less direct control* over each step's contract than in CrewAI/LangGraph.

A second observation: AutoGen's natural unit is the **message**, not the **task**. If you need typed, structured outputs at each step, you'll wrap each agent's output in a parser. If you need free-form back-and-forth, AutoGen is the most natural fit.


---
## 5. LangGraph — explicit state machine

**Mental model**: a **graph** of typed state. You define a `TypedDict` state, write functions that take state and return state-updates, and wire them with `add_edge` / `add_conditional_edges`. This is what you saw in Lesson 11.

**Pros**
- Most **control** of the three. You see every transition.
- First-class **persistence** via checkpointers — pause an agent mid-graph, resume next week. (We covered this in L11.)
- Strong observability (LangSmith) and `streaming` support.
- Production-friendly: explicit retries, time-travel, human-in-the-loop.

**Cons**
- Lowest level → most code per feature. Closer to "you wrote it" than "framework wrote it".
- Closer to a state-machine library than a multi-agent library. You build "multi-agent" by making each agent a node.

We'll do a compact version (you've seen the full pattern in L11).


In [ ]:
from langgraph.graph import StateGraph, END, START
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

lc_llm = ChatAnthropic(model=MODEL, temperature=0.3, max_tokens=1024)

class State(TypedDict):
    topic: str
    draft: str
    critique: str
    iteration: int

def node_researcher(state: State) -> Dict[str, Any]:
    iteration = state["iteration"] + 1
    user = (
        f"TOPIC: {state['topic']}\n\nPRIOR DRAFT:\n{state['draft'] or '(none)'}\n\n"
        f"PRIOR CRITIQUE TO ADDRESS:\n{state['critique'] or '(none)'}\n\n"
        "Write a ~150-word brief. If a prior critique exists, revise to address it. No preamble."
    )
    resp = lc_llm.invoke([
        SystemMessage(content="You write concise technical briefs for Java engineers new to AI."),
        HumanMessage(content=user),
    ])
    return {"draft": resp.content, "iteration": iteration}

def node_critic(state: State) -> Dict[str, Any]:
    resp = lc_llm.invoke([
        SystemMessage(content=(
            "You critique briefs. Reply EXACTLY 'APPROVE' if excellent for a Java engineer new to AI; "
            "otherwise list 2-3 specific edit requests. No other text."
        )),
        HumanMessage(content=f"BRIEF:\n{state['draft']}"),
    ])
    return {"critique": resp.content}

def route(state: State) -> str:
    if "APPROVE" in state["critique"].upper() or state["iteration"] >= MAX_ITERATIONS:
        return END
    return "researcher"

g = StateGraph(State)
g.add_node("researcher", node_researcher)
g.add_node("critic", node_critic)
g.add_edge(START, "researcher")
g.add_edge("researcher", "critic")
g.add_conditional_edges("critic", route, {"researcher": "researcher", END: END})
app = g.compile()

lg_final = app.invoke({"topic": TOPIC, "draft": "", "critique": "", "iteration": 0})

print(f"=== LANGGRAPH RESULT ===")
print(f"iterations : {lg_final['iteration']}")
print(f"critique   : {lg_final['critique'][:120]}...")
print(f"\nFINAL BRIEF:\n{lg_final['draft']}")

# Visualize the graph (optional, requires graphviz; mermaid str works in any env).
try:
    print("\n--- Mermaid diagram ---")
    print(app.get_graph().draw_mermaid())
except Exception as e:
    print("(graph viz unavailable:", e, ")")

# 💡 EXPERIMENT: add a `checkpointer=MemorySaver()` on compile() and run with
#                config={'configurable': {'thread_id': '...'}}. Now you can resume.
# 💡 EXPERIMENT: add a parallel `fact_checker` node that runs in parallel with `critic`
#                and joins back via a `merge` node. LangGraph excels at fan-out/fan-in.


**What you should observe**: LangGraph forced you to spell out every transition (`add_edge`, `add_conditional_edges`). The benefit is total transparency — you can read the graph and *know* what will happen. The cost is more code.

You also got, for free, a visualization of the state machine — that's a real production win when explaining your agent to a non-technical PM or a security reviewer.


---
## 6. Side-by-side: the same loop, three philosophies

Here's the code shape of each, distilled:

```python
# CrewAI:  describe roles + tasks
crew = Crew(agents=[researcher, critic], tasks=[research_task, crit_task], process=Process.sequential)
crew.kickoff()                                        # loop you wrote in Python around it

# AutoGen: list participants + stop condition
team = RoundRobinGroupChat([researcher, critic],
                           termination_condition=TextMentionTermination("APPROVE"))
await team.run(task=topic)                            # framework runs the loop

# LangGraph: draw the state machine yourself
g.add_edge(START, "researcher")
g.add_edge("researcher", "critic")
g.add_conditional_edges("critic", route, {"researcher": "researcher", END: END})
app.invoke({...})                                     # graph executes
```

The **same loop**, three different things at the center of the abstraction:

| Center | Strength | Weakness |
|---|---|---|
| **CrewAI: roles & tasks** | Reads like a project plan | Loops & branching are awkward |
| **AutoGen: conversation** | Emergent, code-exec-friendly | Less direct control per step |
| **LangGraph: state machine** | Total control, persistence | More code |

There is no "right" — they make different trade-offs explicit.


---
## 7. Tool use — how each framework attaches tools

All three frameworks let your agent call tools. The shape differs.

### CrewAI
- Built-in `crewai_tools` library (web search, file ops, etc.).
- Decorate a plain function with `@tool` (from `crewai.tools`) and pass via `tools=[...]` on `Agent`.

### AutoGen
- Pass a list of plain Python functions to `AssistantAgent(tools=[fn1, fn2])`. The framework derives schemas from type hints + docstring.
- For code execution: `UserProxyAgent(code_executor=DockerCommandLineCodeExecutor(...))`.

### LangGraph
- Use `langchain_core.tools.tool` decorator, or pre-built tools, then bind to the LLM: `llm.bind_tools([t])`. Add a `ToolNode` in the graph.

A tiny tool the researcher could call in any of the three:


In [ ]:
# Define one tool, three ways to mount it.

def get_topic_keywords(topic: str) -> str:
    """Return 3-5 keywords associated with a topic (mock — would be a real search in prod)."""
    # Keep it deterministic for the demo.
    canned = {
        "tool use in llm agents": "function-calling, ReAct loop, tool registry, schema",
        "rag":                    "embeddings, retrieval, chunking, hybrid search",
        "default":                "agent, LLM, system prompt, evaluation, deployment",
    }
    key = topic.lower().strip()
    return canned.get(key, canned["default"])

# --- AutoGen: pass plain functions ---
ag_researcher_with_tools = AssistantAgent(
    name="researcher_with_tools",
    model_client=model_client,
    system_message="Before writing, call get_topic_keywords to anchor your brief. Then write.",
    tools=[get_topic_keywords],
)
async def demo_autogen_tools():
    team_t = RoundRobinGroupChat(
        [ag_researcher_with_tools],
        termination_condition=MaxMessageTermination(4),
    )
    return await team_t.run(task=f"Topic: {TOPIC}. Write the brief.")

at = await demo_autogen_tools()
print("--- AutoGen with tool ---")
for m in at.messages[-3:]:
    src = getattr(m, "source", "?")
    content = getattr(m, "content", str(m))
    if isinstance(content, list):
        content = " ".join(str(c) for c in content)
    print(f"[{src}] {str(content)[:200]}")

# --- LangGraph: bind & ToolNode (sketch only — full version covered in L4/L11) ---
from langchain_core.tools import tool as lc_tool
@lc_tool
def get_topic_keywords_lc(topic: str) -> str:
    """Return 3-5 keywords associated with a topic."""
    return get_topic_keywords(topic)

lc_llm_with_tools = lc_llm.bind_tools([get_topic_keywords_lc])
resp = lc_llm_with_tools.invoke([HumanMessage(content=f"Find keywords for: {TOPIC}, then summarize them.")])
print("\n--- LangGraph LLM with bound tool ---")
print("tool_calls:", getattr(resp, "tool_calls", None))
print("content   :", str(resp.content)[:200])

# --- CrewAI: @tool decorator + agent tools= (sketch — full chain in CrewAI docs) ---
from crewai.tools import tool as crew_tool

@crew_tool("topic_keywords")
def topic_keywords_tool(topic: str) -> str:
    """Return 3-5 keywords associated with a topic."""
    return get_topic_keywords(topic)

researcher_with_tool = Agent(
    role="Researcher",
    goal="Use the keyword tool to anchor your brief, then write.",
    backstory="Methodical, tool-using analyst.",
    llm=llm,
    tools=[topic_keywords_tool],
    verbose=False,
)
print("\n--- CrewAI tool wiring ---")
print("Agent tools registered:", [t.name for t in researcher_with_tool.tools])


**Takeaway**: every framework gives you a way to attach tools, and the underlying mechanism is the same (function-calling on the LLM). What differs is how *opinionated* the wrapper is. AutoGen's "just pass plain functions" is the gentlest learning curve; LangGraph's `ToolNode` gives you the most production primitives (retries, parallel tools, structured tool errors).


---
## 8. Decision framework — when to pick which

You should rarely pick a framework on "vibes". Use this checklist when starting a new agent project:

### Pick **LangGraph** when…
- You need **production-grade control**: explicit retries, persistence, time-travel, observability.
- The agent must **resume from interruption** (long-running tasks, human-in-the-loop).
- You want **streaming output** with per-node events (we did this in L19).
- Your team is comfortable reading a state machine.
- You're integrating with LangSmith, LangChain ecosystem.

### Pick **CrewAI** when…
- The workflow is **roles doing tasks in order** ("research → write → edit → publish").
- You want to **prototype fast** — non-engineers on the team can read the agent code.
- You want a **manager agent** to delegate (`Process.hierarchical`).
- Heavy looping / conditional branching is NOT the primary pattern.

### Pick **AutoGen** when…
- You want **emergent multi-agent conversation** — debate, brainstorm, code-and-fix.
- Your agents need to **execute code** safely (`DockerCommandLineCodeExecutor`).
- You like **async-native** + streaming agent events.
- You can tolerate (and observe) some emergent behavior.

### Build **bespoke** (your own loop, like L4) when…
- Your agent is a **single tool-use ReAct loop** — frameworks are overkill.
- Latency is critical and you can't afford framework overhead.
- You're learning — once you've hand-built it, you'll understand the frameworks far better. **This is exactly what we've been doing in Lessons 1-20.**
- You have unusual constraints (custom transport, exotic tool protocols, MCP-only) where the framework abstractions get in the way.

### A useful filter: ask "where does the **state** live?"
- LangGraph → in the **state dict**, explicit, you defined it.
- CrewAI → in the **task outputs**, semi-implicit.
- AutoGen → in the **message history**, fully implicit.

The more you need to *reason about* and *test* the state, the more you want LangGraph. The more you want the LLM to *figure things out together*, the more you want AutoGen.


---
## 9. Mini-Capstone — choose your stack for AutoResearcher

By Lesson 23 you'll ship AutoResearcher v1.0. Time to make a call: **which framework does the orchestration?**

Here's the recommendation tree for your specific situation:

1. **Hard requirements you've already accumulated**:
   - Streaming agent events (L19 ✅) → favors **LangGraph** or **AutoGen**.
   - Persistence across runs (we built a `MemorySaver` mental model in L11) → strongly favors **LangGraph**.
   - Eval-gated CI/CD (L17 ✅) → all three work, but LangGraph's per-node observability makes it easiest.
   - MCP integration (L14 ✅) → all three can call MCP tools via a small bridge.
2. **What you'd lose by going CrewAI**: tight self-correcting loops; explicit state. Not great for an iterative research agent.
3. **What you'd lose by going AutoGen**: predictable, contract-driven outputs; visible state.
4. **What you'd lose by going LangGraph**: a tiny bit of velocity.

For AutoResearcher, **LangGraph is the natural fit**. Why:
- The agent has clear states: `research → reflect → revise → verify → finalize`. State machine is the right abstraction.
- It needs to **persist** intermediate research so the user can resume.
- We've already done a Phase-2 version in L11 — we'll extend, not rewrite.

But — and this matters — **you've also built the bespoke ReAct loop yourself in L4**. You could ship AutoResearcher as a *pure-Python* agent with no framework. The result would be smaller, faster, and easier to debug than any framework. It depends on whether you want the *open-source value-add* to be "look at this neat agent" or "look at this neat framework usage."

Below is a one-shot exercise: take the CrewAI version of the critic-loop above and port it to LangGraph (almost a one-liner since the code is already separate). Then think about which one feels easier to debug.


In [ ]:
# 💡 EXERCISE — pick a topic that interests you and run the SAME task in two frameworks.
# Then compare:
#   1. How many lines of code did each take?
#   2. How easy is it to add a third agent (e.g., a fact-checker)?
#   3. How easy is it to swap in a different LLM (e.g., for cost optimization, L22)?
#   4. How easy is it to persist progress and resume from a crash?

YOUR_TOPIC = "Why latency matters in production LLM agents and how to measure it"

# Try CrewAI:
your_crewai = run_crewai_loop(YOUR_TOPIC, max_iter=2)
print("CrewAI iterations :", your_crewai["iterations"])
print(your_crewai["final_draft"][:300], "...\n")

# Try LangGraph:
your_lg = app.invoke({"topic": YOUR_TOPIC, "draft": "", "critique": "", "iteration": 0})
print("LangGraph iterations :", your_lg["iteration"])
print(your_lg["draft"][:300], "...")

# 💡 EXPERIMENT: add a 'fact_checker' role/node in both, then count the lines of code
#                you needed to add. The result will inform your AutoResearcher choice.


---
## 10. Recap & what's next

You now have working code, side-by-side, for three of the major agent frameworks in 2026:

- **CrewAI** — roles + tasks + sequential / hierarchical process. Fast prototype.
- **AutoGen** (v0.4+) — conversational agents + termination. Emergent collaboration.
- **LangGraph** — explicit state machine. Production control.

You also have a **decision framework** that lets you pick *with a reason* — and the awareness that "build bespoke" is a perfectly valid 4th option you've already practiced.

### A few cultural notes that matter in real teams
1. **Framework lock-in is real, but smaller than it looks**. All three call the same LLM APIs underneath. A migration is mostly re-wiring agent definitions; the prompts move 1:1.
2. **Read the framework's source for one feature you care about**. You'll learn whether the framework treats tools/state/streaming as first-class or bolted-on.
3. **Frameworks are not always the answer**. The most production-stable AI agents at large labs (yes, including some of ours) are bespoke loops on top of typed state. We've taught you those skills first for that reason.

### What's next (Lesson 22 preview)
**Cost Engineering** — model routing (small model for cheap turns, big model for hard turns), prompt caching (Anthropic supports cache_control breakpoints — huge wins on system prompts that don't change), batching, token compression, and per-tenant cost observability. By the end of L22 you'll be ready to make AutoResearcher 5-10× cheaper without sacrificing quality.

### Open-source angle
The most useful artifact from today's notebook is the **decision tree + side-by-side code** you ran. That's already a small blog post or `README.md` chapter that engineers shipping their first agent will star. Worth saving the LangGraph version of the loop and growing it into the AutoResearcher backbone.

Tomorrow we sharpen the cost knife. See you then, Gourav.
